In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 1: Install packages and authenticate
# ==============================================================

!pip install earthengine-api --quiet

import ee
import time
import json
import os
import sys
from datetime import datetime

PROJECT = 'osmgee'

try:
    ee.Initialize(project=PROJECT)
    _auth = 'already authenticated'
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    _auth = 'authenticated now'

print("")
print("=" * 55)
print("CELL 1 COMPLETE")
print("   Packages installed")
print("   Earth Engine: " + _auth)
print("   Project: osmgee")
print("")
print(">>> RUN CELL 2 NEXT")
print("=" * 55)


CELL 1 COMPLETE
   Packages installed
   Earth Engine: authenticated now
   Project: osmgee

>>> RUN CELL 2 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 2: Local state (no Drive)
# ==============================================================
#
# Drive is NOT needed. Resume capability comes from querying GEE
# directly — which assets exist, and which tasks are running.
# Drive would only cache tile areas (~1 min to recompute) and keep
# an audit log. Neither is required.
#
# These paths live in Colab's temporary storage and vanish when the
# session ends. That is fine: Cell 8 rebuilds the full state from
# Earth Engine every time.

STATE_DIR  = '/content/ccdc_state'
os.makedirs(STATE_DIR, exist_ok=True)

LOG_FILE   = STATE_DIR + '/submission_log.json'
AREA_CACHE = STATE_DIR + '/tile_areas.json'

print("State directory:", STATE_DIR, "(temporary)")
print("")
print("NOTE: resume does NOT depend on this. After a disconnect,")
print("      re-run Cells 1-8 and the state is rebuilt from GEE.")

print("")
print("=" * 55)
print("CELL 2 COMPLETE")
print("   Local state directory ready")
print("   No Drive access needed")
print("")
print(">>> RUN CELL 3 NEXT")
print("=" * 55)

State directory: /content/ccdc_state (temporary)

NOTE: resume does NOT depend on this. After a disconnect,
      re-run Cells 1-8 and the state is rebuilt from GEE.

CELL 2 COMPLETE
   Local state directory ready
   No Drive access needed

>>> RUN CELL 3 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 3: Configuration
# ==============================================================

CONFIG = {
    'AOI_ASSET':      'projects/osmgee/assets/HUC8_Dissolved',
    'ASSET_PROJECT':  'projects/osmgee/assets',
    'ASSET_PREFIX':   'ccdc_dense_',

    'START_YEAR': 1985,          # 1984 excluded: L5 launch year, sparse
    'END_YEAR':   2025,

    'TARGET_TILES':    650,
    'TILE_SIZE_FIXED': None,
    'GRID_CRS':        'EPSG:32612',

    # CCDC parameters — GEE operator defaults, not Zhu & Woodcock values
    'MIN_OBSERVATIONS':  6,
    'CHI_SQUARE_PROB':   0.99,
    'MIN_YEARS_SCALER':  1.33,
    'LAMBDA':            0.002,
    'MAX_ITERATIONS':    10000,
    'DATE_FORMAT':       2,      # unix milliseconds

    'CLOUD_THRESHOLD':     60,
    'MAX_IMAGES_PER_TILE': 4000,
    'MASK_SNOW':           True,
    'HARMONIZE_L89':       False,

    'NODATA':       -9999,
    'EXPORT_SCALE': 30,
    'EXPORT_CRS':   'EPSG:32612',

    # reliability thresholds for the final segment
    'REL_MIN_LAST_SEG_YEARS': 1.0,
    'REL_MIN_OBS_LAST':       12,
    'REL_MAX_ABS_SLOPE':      0.05,
    'REL_MAX_AMPLITUDE':      0.5,
    'REL_MAX_RMSE':           0.3,
    'REL_MAX_BREAKS':         8,

    'MIN_TILE_FRACTION': 0.02,

    'MAX_CONCURRENT': 40,
    'POLL_SECONDS':   120,
    'RETRY_TRIES':    3,
    'RETRY_DELAY':    10,
}

MILLIS_PER_YEAR = 31557600000
EPOCH_YEAR = 1970
ND = CONFIG['NODATA']

BREAKPOINT_BANDS = ['green', 'red', 'nir', 'swir1', 'swir2']
TMASK_BANDS      = ['green', 'swir2']
INPUT_BANDS      = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2',
                    'NDVI', 'NBR', 'NDMI', 'EVI']
TM  = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
OLI = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']


def asset_path(name):
    return CONFIG['ASSET_PROJECT'] + '/' + CONFIG['ASSET_PREFIX'] + name


def retry(fn, tries=None, delay=None):
    """Exponential backoff. GEE calls fail transiently and one bad call
    should not kill a loop with hours of work remaining."""
    tries = tries or CONFIG['RETRY_TRIES']
    delay = delay or CONFIG['RETRY_DELAY']
    last = None
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if i < tries - 1:
                time.sleep(delay * (2 ** i))
    raise last

print("Period      :", CONFIG['START_YEAR'], "-", CONFIG['END_YEAR'])
print("Target tiles:", CONFIG['TARGET_TILES'])
print("Export CRS  :", CONFIG['EXPORT_CRS'], "at", CONFIG['EXPORT_SCALE'], "m")
print("Harmonize   :", CONFIG['HARMONIZE_L89'], "(measured: raw C2 is best)")

print("")
print("=" * 55)
print("CELL 3 COMPLETE")
print("   Config loaded")
print("   Constants and helpers defined")
print("")
print(">>> RUN CELL 4 NEXT")
print("=" * 55)

Period      : 1985 - 2025
Target tiles: 650
Export CRS  : EPSG:32612 at 30 m
Harmonize   : False (measured: raw C2 is best)

CELL 3 COMPLETE
   Config loaded
   Constants and helpers defined

>>> RUN CELL 4 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 4: Landsat preprocessing functions
# ==============================================================

def cloud_mask(img):
    """QA_PIXEL bits: fill, dilated cloud, cirrus, cloud, shadow, snow.
    Plus QA_RADSAT saturation. Snow matters in boreal Canada."""
    qa = img.select('QA_PIXEL')
    m = (qa.bitwiseAnd(1 << 0).eq(0)
         .And(qa.bitwiseAnd(1 << 1).eq(0))
         .And(qa.bitwiseAnd(1 << 2).eq(0))
         .And(qa.bitwiseAnd(1 << 3).eq(0))
         .And(qa.bitwiseAnd(1 << 4).eq(0)))
    if CONFIG['MASK_SNOW']:
        m = m.And(qa.bitwiseAnd(1 << 5).eq(0))
    return img.updateMask(m).updateMask(img.select('QA_RADSAT').eq(0))


def harmonize(img, bands):
    """Scale to surface reflectance. L7/L8 harmonization is OFF by
    MEASUREMENT: over 939 L7 and 1005 L8 scenes the mean absolute bias
    was 0.0023 for raw Collection 2 against 0.0089 and 0.0124 for the
    two Roy et al. (2016) transforms, which were derived from
    Collection 1 and make C2 worse."""
    sr = (img.select(bands, ['blue', 'green', 'red', 'nir', 'swir1', 'swir2'])
          .multiply(0.0000275).add(-0.2).toFloat())
    if CONFIG['HARMONIZE_L89']:
        raise NotImplementedError(
            'Harmonization is off by measurement, not oversight.')
    sr = sr.updateMask(sr.reduce(ee.Reducer.min()).gt(-0.2)
                       .And(sr.reduce(ee.Reducer.max()).lt(1.6)))
    return sr.copyProperties(img, ['system:time_start'])


def add_indices(img):
    ndvi = img.normalizedDifference(['nir', 'red']).rename('NDVI')
    nbr  = img.normalizedDifference(['nir', 'swir2']).rename('NBR')
    ndmi = img.normalizedDifference(['nir', 'swir1']).rename('NDMI')
    evi  = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': img.select('nir'), 'RED': img.select('red'),
         'BLUE': img.select('blue')}).clamp(-1, 1).rename('EVI')
    return img.addBands([ndvi, nbr, ndmi, evi])


def load_collection(geom):
    """L5/L7/L8/L9 Collection 2 Level-2. Note filterDate end is
    EXCLUSIVE, so END_YEAR+1 is used to include the final day."""
    end_ex = str(CONFIG['END_YEAR'] + 1) + '-01-01'

    def f(cid, bands, d0, d1):
        return (ee.ImageCollection(cid)
                .filterBounds(geom).filterDate(d0, d1)
                .filter(ee.Filter.lt('CLOUD_COVER', CONFIG['CLOUD_THRESHOLD']))
                .map(cloud_mask)
                .map(lambda i: harmonize(i, bands))
                .map(add_indices))

    return (f('LANDSAT/LT05/C02/T1_L2', TM,
              str(CONFIG['START_YEAR']) + '-01-01', '2012-05-06')
            .merge(f('LANDSAT/LE07/C02/T1_L2', TM, '1999-01-01', '2022-04-07'))
            .merge(f('LANDSAT/LC08/C02/T1_L2', OLI, '2013-03-18', end_ex))
            .merge(f('LANDSAT/LC09/C02/T1_L2', OLI, '2021-10-31', end_ex))
            .sort('system:time_start'))

print("")
print("=" * 55)
print("CELL 4 COMPLETE")
print("   cloud_mask defined")
print("   harmonize defined")
print("   add_indices defined")
print("   load_collection defined")
print("")
print(">>> RUN CELL 5 NEXT")
print("=" * 55)


CELL 4 COMPLETE
   cloud_mask defined
   harmonize defined
   add_indices defined
   load_collection defined

>>> RUN CELL 5 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 5: Extraction — CCDC_RESULTS.extract()
# ==============================================================
#
# Line-by-line translation of CCDC_RESULTS.extract() in v5.7.
# Every numbered fix is preserved and labelled so it can be traced
# across both implementations.
#
# Produces 39 output bands.

def extract(ccdc, geom):
    t_break = ccdc.select('tBreak')
    t_start = ccdc.select('tStart')
    t_end = ccdc.select('tEnd')

    # FIX 1 — undisturbed pixels store tBreak = [0], so arrayLength
    # returns 1 and the naive test passes everywhere.
    has_break = t_break.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0]).gt(0)

    # FIX 3 — count real breaks, not segments
    num_breaks = (t_break.gt(0).arrayReduce(ee.Reducer.sum(), [0])
                  .arrayGet([0]).toFloat())
    num_segments = t_end.arrayLength(0).toFloat()

    def ms_to_year(img):
        return img.divide(MILLIS_PER_YEAR).add(EPOCH_YEAR)

    # FIX 2 — clamp real values FIRST, then mask, then sentinel. The
    # reverse order mapped every undisturbed pixel onto the record start.
    def year_band(ms_img, name):
        return (ms_to_year(ms_img)
                .clamp(CONFIG['START_YEAR'], CONFIG['END_YEAR'])
                .updateMask(has_break).unmask(ND).toFloat().rename(name))

    first_break = year_band(t_break.arrayGet(0), 'first_break_year')
    last_break = year_band(t_break.arraySlice(0, -1, None).arrayGet([0]),
                           'last_break_year')
    change_count = num_breaks.rename('change_count')

    first_start = (ms_to_year(t_start.arrayGet(0))
                   .clamp(CONFIG['START_YEAR'], CONFIG['END_YEAR'])
                   .toFloat().rename('first_segment_start'))
    last_end = (ms_to_year(t_end.arraySlice(0, -1, None).arrayGet([0]))
                .clamp(CONFIG['START_YEAR'], CONFIG['END_YEAR'])
                .toFloat().rename('last_segment_end'))
    # FIX 11 — the last segment's OWN start
    last_start = (ms_to_year(t_start.arraySlice(0, -1, None).arrayGet([0]))
                  .clamp(CONFIG['START_YEAR'], CONFIG['END_YEAR'])
                  .toFloat().rename('last_segment_start'))

    ndvi_coefs = ccdc.select('NDVI_coefs')
    nbr_coefs = ccdc.select('NBR_coefs')

    # FIX 8 — keep coefficients in DOUBLE. Casting to float32 here caused
    # catastrophic cancellation: the intercept is the value at 1970, so a
    # recent segment can legitimately have an intercept near -458 and
    # fitted = -458.37 + 458.90 loses all precision in float32.
    def last_coef(c, i):
        return c.arraySlice(0, -1, None).arrayGet([0, i])

    def first_coef(c, i):
        return c.arrayGet([0, i])

    icpt_last = last_coef(ndvi_coefs, 0)
    slope_last = last_coef(ndvi_coefs, 1).multiply(MILLIS_PER_YEAR)
    icpt_first = first_coef(ndvi_coefs, 0)
    slope_first = first_coef(ndvi_coefs, 1).multiply(MILLIS_PER_YEAR)

    ndvi_intercept = icpt_last.toFloat().rename('ndvi_intercept_last')
    ndvi_slope = slope_last.toFloat().rename('ndvi_slope_last')
    ndvi_slope_fst = slope_first.toFloat().rename('ndvi_slope_first')

    c1, s1 = last_coef(ndvi_coefs, 2), last_coef(ndvi_coefs, 3)
    c2, s2 = last_coef(ndvi_coefs, 4), last_coef(ndvi_coefs, 5)
    c3, s3 = last_coef(ndvi_coefs, 6), last_coef(ndvi_coefs, 7)

    nbr_intercept = last_coef(nbr_coefs, 0).toFloat().rename('nbr_intercept_last')
    nbr_slope = (last_coef(nbr_coefs, 1).multiply(MILLIS_PER_YEAR)
                 .toFloat().rename('nbr_slope_last'))

    ann_amp = c1.hypot(s1).toFloat().rename('ndvi_annual_amplitude')
    ann_phs = s1.atan2(c1).toFloat().rename('ndvi_annual_phase')
    semi_amp = c2.hypot(s2).toFloat().rename('ndvi_semiannual_amplitude')
    tert_amp = c3.hypot(s3).toFloat().rename('ndvi_tertiary_amplitude')

    def mag_band(band, idx, name):
        if idx == 'last':
            img = ccdc.select(band).arraySlice(0, -1, None).arrayGet([0])
        else:
            img = ccdc.select(band).arrayGet(idx)
        return img.updateMask(has_break).unmask(ND).toFloat().rename(name)

    ndvi_mag_first = mag_band('NDVI_magnitude', 0, 'ndvi_mag_first')
    ndvi_mag_last = mag_band('NDVI_magnitude', 'last', 'ndvi_mag_last')
    nbr_mag_first = mag_band('NBR_magnitude', 0, 'nbr_mag_first')

    # FIX 23 — the TRUE last-break probability by dynamic indexing.
    # tBreak fills indices 0..numBreaks-1, so index numBreaks-1 is the
    # last ACTUAL break. The final SEGMENT's value is normally ~0.
    last_break_idx = num_breaks.subtract(1).max(0).toInt()
    cp_first = (ccdc.select('changeProb').arrayGet(0)
                .updateMask(has_break).unmask(ND).toFloat()
                .rename('change_probability_first'))
    cp_last = (ccdc.select('changeProb').arrayGet(last_break_idx)
               .updateMask(has_break).unmask(ND).toFloat()
               .rename('change_probability_last'))
    # FIX 38 — NOT masked to ND: defined for every pixel, unlike siblings
    cp_final_seg = (ccdc.select('changeProb').arraySlice(0, -1, None)
                    .arrayGet([0]).toFloat()
                    .rename('change_probability_final_segment'))

    ndvi_rmse = (ccdc.select('NDVI_rmse').arraySlice(0, -1, None)
                 .arrayGet([0]).toFloat().rename('ndvi_rmse_last'))
    nbr_rmse = (ccdc.select('NBR_rmse').arraySlice(0, -1, None)
                .arrayGet([0]).toFloat().rename('nbr_rmse_last'))

    num_obs_last = (ccdc.select('numObs').arraySlice(0, -1, None)
                    .arrayGet([0]).toFloat().rename('num_obs_last'))
    num_obs_tot = (ccdc.select('numObs').arrayReduce(ee.Reducer.sum(), [0])
                   .arrayGet([0]).toFloat().rename('num_obs_total'))

    # FIX 13 — this is total record coverage, not one segment's duration
    duration = last_end.subtract(first_start).max(0).toFloat().rename(
        'time_coverage_years')
    last_seg_dur = last_end.subtract(last_start).max(0).toFloat().rename(
        'last_segment_duration_years')
    obs_per_year = num_obs_tot.divide(duration.max(1)).toFloat().rename(
        'obs_per_year')

    valid_last = last_break.neq(ND)
    time_since = (ee.Image.constant(CONFIG['END_YEAR']).subtract(last_break)
                  .updateMask(valid_last).unmask(ND).toFloat()
                  .rename('years_since_disturbance'))

    valid_mag = ndvi_mag_first.neq(ND)
    change_type = (ndvi_mag_first.lt(0).updateMask(valid_mag).unmask(ND)
                   .toFloat().rename('change_type_loss'))
    severity = (ndvi_mag_first.abs().updateMask(valid_mag).unmask(ND)
                .toFloat().rename('disturbance_severity'))

    # FIX 22 — trend component at each segment's MIDPOINT. A Fourier
    # series integrates to exactly zero over a full period, so
    # intercept + slope*t is the exact mean over the year CENTRED on t.
    # Evaluating at a segment endpoint puts half the averaging window
    # outside the fitted span.
    # FIX 11 — each model is evaluated only inside its own segment.
    mid_last = last_start.add(last_end).divide(2)
    first_end = (ms_to_year(t_end.arrayGet(0))
                 .clamp(CONFIG['START_YEAR'], CONFIG['END_YEAR']))
    mid_first = first_start.add(first_end).divide(2)

    trend_last_raw = icpt_last.add(
        slope_last.multiply(mid_last.subtract(EPOCH_YEAR)))
    trend_first_raw = icpt_first.add(
        slope_first.multiply(mid_first.subtract(EPOCH_YEAR)))

    # FIX 9 — clamp to the physical NDVI range. A real bound on the
    # quantity, not circular: qa_flags below uses the UNCLAMPED value.
    trend_last = trend_last_raw.clamp(-1, 1).toFloat().rename(
        'ndvi_trend_component_last')
    trend_first = trend_first_raw.clamp(-1, 1).toFloat().rename(
        'ndvi_trend_component_first')
    trend_change = trend_last.subtract(trend_first).toFloat().rename(
        'ndvi_trend_component_change')
    trend_span = mid_last.subtract(mid_first).max(0.5).toFloat().rename(
        'trend_component_span_years')

    change_freq = (change_count.divide(duration.max(1)).multiply(10)
                   .toFloat().rename('change_frequency_per_decade'))

    # FIX 5 + 14 + 31 — bitmask from UNCLAMPED values, 8 bits
    qa = (trend_last_raw.abs().gt(1.0).multiply(1)
          .add(ndvi_slope.abs().gt(CONFIG['REL_MAX_ABS_SLOPE']).multiply(2))
          .add(ndvi_rmse.gt(CONFIG['REL_MAX_RMSE']).multiply(4))
          .add(obs_per_year.lt(3).multiply(8))
          .add(last_seg_dur.lt(CONFIG['REL_MIN_LAST_SEG_YEARS']).multiply(16))
          .add(num_obs_last.lt(CONFIG['REL_MIN_OBS_LAST']).multiply(32))
          .add(ann_amp.gt(CONFIG['REL_MAX_AMPLITUDE']).multiply(64))
          .add(change_count.gt(CONFIG['REL_MAX_BREAKS']).multiply(128))
          .toUint8().rename('qa_flags'))

    # FIX 30 — raw coefficients exported UNCHANGED; reliability stated
    # separately. Masking would destroy the evidence of a diverged fit.
    reliable = (last_seg_dur.gte(CONFIG['REL_MIN_LAST_SEG_YEARS'])
                .And(num_obs_last.gte(CONFIG['REL_MIN_OBS_LAST']))
                .And(ndvi_slope.abs().lte(CONFIG['REL_MAX_ABS_SLOPE']))
                .And(ann_amp.lte(CONFIG['REL_MAX_AMPLITUDE']))
                .And(ndvi_rmse.lte(CONFIG['REL_MAX_RMSE']))
                .toUint8().rename('last_segment_reliable'))

    out = ee.Image.cat([
        first_break, last_break, change_count,
        num_segments.rename('segment_count'),
        first_start, last_start, last_end, duration, last_seg_dur,
        ndvi_mag_first, ndvi_mag_last, nbr_mag_first,
        cp_first, cp_last, cp_final_seg,
        ndvi_intercept, ndvi_slope, ndvi_slope_fst, nbr_intercept, nbr_slope,
        ann_amp, ann_phs, semi_amp, tert_amp,
        ndvi_rmse, nbr_rmse, num_obs_last, num_obs_tot, obs_per_year,
        time_since, change_type, severity, change_freq,
        trend_first, trend_last, trend_change, trend_span,
        reliable, qa
    ]).clip(geom)

    return (out.toFloat()
            .addBands(qa, None, True)
            .addBands(reliable, None, True))

print("")
print("=" * 55)
print("CELL 5 COMPLETE")
print("   extract() defined")
print("   All numbered fixes preserved")
print("   39 output bands")
print("")
print(">>> RUN CELL 6 NEXT")
print("=" * 55)


CELL 5 COMPLETE
   extract() defined
   All numbered fixes preserved
   39 output bands

>>> RUN CELL 6 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 6: Build grid and cache tile areas
# ==============================================================
#
# Computes every tile's clipped area in ONE server call. An earlier
# design made two getInfo() calls per tile — 1,334 round trips, each
# a chance to hang.

_cache = {}


def build_grid():
    if 'lst' in _cache:
        return _cache['lst'], _cache['size'], _cache['n']

    region = ee.FeatureCollection(CONFIG['AOI_ASSET']).geometry()
    area_km2 = retry(lambda: region.area(1000).divide(1e6).getInfo())

    if CONFIG['TILE_SIZE_FIXED']:
        size = CONFIG['TILE_SIZE_FIXED']
    else:
        # 1.18 accounts for partial tiles along the watershed boundary
        size = (area_km2 * 1e6 * 1.18 / CONFIG['TARGET_TILES']) ** 0.5

    def mk(s):
        proj = ee.Projection(CONFIG['GRID_CRS']).atScale(s)
        return region.bounds().coveringGrid(proj).filterBounds(region)

    grid = mk(size)
    n = retry(lambda: grid.size().getInfo())

    if (not CONFIG['TILE_SIZE_FIXED'] and
            abs(n - CONFIG['TARGET_TILES']) > CONFIG['TARGET_TILES'] * 0.12):
        size *= (n / CONFIG['TARGET_TILES']) ** 0.5
        grid = mk(size)
        n = retry(lambda: grid.size().getInfo())

    lst = grid.toList(grid.size())
    n = retry(lambda: lst.length().getInfo())

    _cache.update({'lst': lst, 'size': size, 'n': n,
                   'region': region, 'area': area_km2})
    return lst, size, n


def tile_areas():
    """Clipped area for every tile, in ONE call, cached to disk."""
    if 'areas' in _cache:
        return _cache['areas']

    lst, size, n = build_grid()

    if os.path.exists(AREA_CACHE):
        try:
            with open(AREA_CACHE) as fh:
                areas = json.load(fh)
            if len(areas) == n:
                _cache['areas'] = areas
                print("Loaded", len(areas), "tile areas from cache")
                return areas
        except Exception:
            pass

    region = _cache['region']
    print("Computing", n, "tile areas in one call — about a minute...")
    fc = ee.FeatureCollection(ee.List.sequence(0, n - 1).map(
        lambda k: ee.Feature(None, {
            'a': ee.Feature(lst.get(k)).geometry()
                 .intersection(region, 100).area(100).divide(1e6)})))
    areas = retry(lambda: fc.aggregate_array('a').getInfo())

    with open(AREA_CACHE, 'w') as fh:
        json.dump(areas, fh)
    _cache['areas'] = areas
    print("Cached", len(areas), "tile areas")
    return areas


lst, size, n_tiles = build_grid()
areas = tile_areas()

full_km2 = (size / 1000) ** 2
n_sliver = sum(1 for a in areas if a / full_km2 < CONFIG['MIN_TILE_FRACTION'])

print("")
print("Watershed :", round(_cache['area']), "km2")
print("Tile size :", round(size), "m  (" + str(round(size/1000, 1)) + " km)")
print("Tiles     :", n_tiles)
print("Slivers   :", n_sliver, "below MIN_TILE_FRACTION, will be skipped")
print("Usable    :", n_tiles - n_sliver)

print("")
print("=" * 55)
print("CELL 6 COMPLETE")
print("   Grid built")
print("   Tile areas cached")
print("   Re-running this cell is instant")
print("")
print(">>> RUN CELL 7 NEXT")
print("=" * 55)

Computing 661 tile areas in one call — about a minute...
Cached 661 tile areas

Watershed : 205423 km2
Tile size : 19311 m  (19.3 km)
Tiles     : 661
Slivers   : 19 below MIN_TILE_FRACTION, will be skipped
Usable    : 642

CELL 6 COMPLETE
   Grid built
   Tile areas cached
   Re-running this cell is instant

>>> RUN CELL 7 NEXT


In [ ]:
# ==============================================================
#     Cell 7b: CCDC call helper — fixes the 'lambda' keyword
# ==============================================================
#
# 'lambda' is reserved in Python, so it cannot be passed as a
# keyword argument, and 'lambda_' is not recognised by the API.
# Dict unpacking sidesteps this: the key is a plain string.

def run_ccdc(col):
    params = {
        'collection':           col,
        'breakpointBands':      BREAKPOINT_BANDS,
        'tmaskBands':           TMASK_BANDS,
        'minObservations':      CONFIG['MIN_OBSERVATIONS'],
        'chiSquareProbability': CONFIG['CHI_SQUARE_PROB'],
        'minNumOfYearsScaler':  CONFIG['MIN_YEARS_SCALER'],
        'dateFormat':           CONFIG['DATE_FORMAT'],
        'lambda':               CONFIG['LAMBDA'],
        'maxIterations':        CONFIG['MAX_ITERATIONS'],
    }
    return ee.Algorithms.TemporalSegmentation.Ccdc(**params)


# quick check that the signature is accepted
_test = run_ccdc(ee.ImageCollection([]))
print("run_ccdc() accepted by the API")

print("")
print("=" * 55)
print("CELL 7b COMPLETE")
print("   run_ccdc() defined")
print("   Cells 7 and 9 must now use it")
print("")
print(">>> RE-RUN CELL 7 WITH THE PATCH BELOW, THEN CELL 9")
print("=" * 55)

run_ccdc() accepted by the API

CELL 7b COMPLETE
   run_ccdc() defined
   Cells 7 and 9 must now use it

>>> RE-RUN CELL 7 WITH THE PATCH BELOW, THEN CELL 9


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 7: Tile task builder
# ==============================================================
#
# Builds ONE export task. Defines only — submits nothing.
#
# Uses run_ccdc() from Cell 7b. 'lambda' is a Python reserved word
# so it cannot be passed as a keyword argument; dict unpacking in
# run_ccdc() sidesteps that.
#
# TWO CHANGES FROM THE JS, both to prevent tile-edge artifacts:
#
#   1. No per-image clip. The export region already defines the
#      output footprint and extract() ends with .clip(geom), so
#      clipping every input image was redundant work that could
#      introduce partial-pixel effects at boundaries.
#
#   2. Loud warning if a tile would be subsampled. Subsampling is
#      the ONLY route by which tile membership can change a result:
#      step size depends on that tile's image count, so neighbouring
#      tiles would get different temporal density and therefore
#      different break sensitivity — a visible seam. Observed range
#      is 1311-1923 images, so MAX_IMAGES_PER_TILE = 4000 should
#      make this impossible. The warning exists so you find out if
#      it ever fires.
#
# Resolution is unaffected: EXPORT_SCALE 30 and EXPORT_CRS
# EPSG:32612 are set on the export itself.

def build_tile_task(idx):
    """Returns (task, n_images) or (None, reason)."""
    lst, size, n = build_grid()
    region = _cache['region']
    areas = tile_areas()

    # sliver rejection, from the cached areas
    full = (size / 1000) ** 2
    frac = areas[idx] / full
    if frac < CONFIG['MIN_TILE_FRACTION']:
        return None, 'sliver, ' + str(round(frac * 100, 1)) + '% of a tile'

    geom = ee.Feature(lst.get(idx)).geometry().intersection(region, 30)
    col = load_collection(geom)
    n_img = retry(lambda: col.size().getInfo())

    if n_img < CONFIG['MIN_OBSERVATIONS']:
        return None, 'only ' + str(n_img) + ' images'

    subsampled = 0
    if n_img > CONFIG['MAX_IMAGES_PER_TILE']:
        subsampled = 1
        print("")
        print("   *** WARNING: tile", idx, "has", n_img, "images,")
        print("   *** above the cap of", CONFIG['MAX_IMAGES_PER_TILE'], ".")
        print("   *** Subsampling changes temporal density relative to")
        print("   *** neighbouring tiles and CAN produce a visible seam.")
        print("   *** Raise MAX_IMAGES_PER_TILE in Cell 3 and resubmit.")
        print("")
        step = -(-n_img // CONFIG['MAX_IMAGES_PER_TILE'])
        ids = list(range(0, n_img, step))
        if ids[-1] != n_img - 1:
            ids.append(n_img - 1)      # keep the most recent observation
        li = col.toList(n_img)
        col = ee.ImageCollection([li.get(i) for i in ids])

    # No per-image clip — see note above
    col = col.select(INPUT_BANDS)

    # <<< the fix: run_ccdc() instead of a direct call with lambda_ >>>
    ccdc = run_ccdc(col)

    # Provenance. The GEE operator cannot be version-pinned and USGS
    # periodically reprocesses Collection 2, so recording what was
    # REQUESTED is the only reproducibility anchor available.
    prov = {
        'script_version':   'ccdc_v5.7_colab',
        'run_date':         datetime.utcnow().strftime('%Y-%m-%d'),
        'period':           str(CONFIG['START_YEAR']) + '-' + str(CONFIG['END_YEAR']),
        'tile_index':       idx,
        'tile_size_m':      int(size),
        'min_observations': CONFIG['MIN_OBSERVATIONS'],
        'chi_square_prob':  CONFIG['CHI_SQUARE_PROB'],
        'min_years_scaler': CONFIG['MIN_YEARS_SCALER'],
        'lambda':           CONFIG['LAMBDA'],
        'date_format':      CONFIG['DATE_FORMAT'],
        'breakpoint_bands': ','.join(BREAKPOINT_BANDS),
        'cloud_threshold':  CONFIG['CLOUD_THRESHOLD'],
        'harmonize_l89':    0,
        'n_images':         n_img,
        'subsampled':       subsampled,
        'note':             'change detection only (CCD), no land-cover step'}

    task = ee.batch.Export.image.toAsset(
        image=extract(ccdc, geom).set(prov),
        description=CONFIG['ASSET_PREFIX'] + 'summary_' + str(idx),
        assetId=asset_path('summary_' + str(idx)),
        region=geom,
        scale=CONFIG['EXPORT_SCALE'],
        crs=CONFIG['EXPORT_CRS'],
        maxPixels=int(1e13),
        pyramidingPolicy={'.default': 'sample'})

    return task, n_img

print("")
print("=" * 55)
print("CELL 7 COMPLETE")
print("   build_tile_task defined")
print("   Uses run_ccdc() — lambda keyword fixed")
print("   No per-image clip (prevents edge effects)")
print("   Warns loudly if any tile would be subsampled")
print("   Nothing submitted yet")
print("")
print(">>> RUN CELL 8 NEXT")
print("=" * 55)


CELL 7 COMPLETE
   build_tile_task defined
   Uses run_ccdc() — lambda keyword fixed
   No per-image clip (prevents edge effects)
   Warns loudly if any tile would be subsampled
   Nothing submitted yet

>>> RUN CELL 8 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 8: State tracking
# ==============================================================
#
# Classifies every tile into one of five states.
#
# ACTIVE is the one that matters for resuming. A tile whose task is
# still RUNNING has no asset yet, so checking assets alone would
# wrongly resubmit it — creating two tasks writing the same asset.
# Checking task state as well is what prevents that.

def completed_tiles():
    """Assets that exist. PAGINATED — one call truncates past ~1000."""
    have, token = set(), None
    pre = CONFIG['ASSET_PREFIX'] + 'summary_'
    for _ in range(50):
        kw = {'parent': CONFIG['ASSET_PROJECT']}
        if token:
            kw['pageToken'] = token
        page = retry(lambda: ee.data.listAssets(kw))
        for a in page.get('assets', []):
            leaf = a['name'].split('/')[-1]
            if leaf.startswith(pre) and leaf[len(pre):].isdigit():
                have.add(int(leaf[len(pre):]))
        token = page.get('nextPageToken')
        if not token:
            break
    return have


def task_states():
    """Tile index -> task state, from the operations list."""
    ops = retry(lambda: ee.data.listOperations())
    pre = CONFIG['ASSET_PREFIX'] + 'summary_'
    out = {}
    for o in ops:
        meta = o.get('metadata', {})
        d = meta.get('description', '')
        if not d.startswith(pre):
            continue
        suf = d[len(pre):]
        if not suf.isdigit():
            continue
        i = int(suf)
        st = meta.get('state', '')
        # if a tile was retried, keep the most informative state
        if out.get(i) != 'SUCCEEDED':
            out[i] = st
    return out


def get_state(verbose=True):
    lst, size, n = build_grid()
    areas = tile_areas()
    full = (size / 1000) ** 2

    done_set = completed_tiles()
    ts = task_states()

    active = {i for i, s in ts.items()
              if s in ('PENDING', 'RUNNING')} - done_set
    failed = {i for i, s in ts.items()
              if s in ('FAILED', 'CANCELLED')} - done_set - active
    sliver = {i for i in range(n)
              if areas[i] / full < CONFIG['MIN_TILE_FRACTION']}
    todo = set(range(n)) - done_set - active - failed - sliver

    if verbose:
        print("  DONE   ", str(len(done_set)).rjust(4), "  asset exists")
        print("  ACTIVE ", str(len(active)).rjust(4), "  task running, no asset yet")
        print("  FAILED ", str(len(failed)).rjust(4), "  task errored")
        print("  SLIVER ", str(len(sliver)).rjust(4), "  too small, skipped")
        print("  TODO   ", str(len(todo)).rjust(4), "  not yet submitted")
        print("  " + "-" * 34)
        print("  TOTAL  ", str(n).rjust(4))
        print("")
        print("  progress:", round(100 * len(done_set) / n, 1), "%")
        if failed and len(failed) <= 40:
            print("  failed indices:", sorted(failed))

    return {'done': done_set, 'active': active, 'failed': failed,
            'sliver': sliver, 'todo': todo, 'n': n}


print("Current state:")
print("")
state = get_state()

print("")
print("=" * 55)
print("CELL 8 COMPLETE")
print("   State tracking defined")
print("   Resuming is now safe")
print("")
print(">>> RUN CELL 9 NEXT")
print("=" * 55)

Current state:

  DONE     642   asset exists
  ACTIVE     0   task running, no asset yet
  FAILED     0   task errored
  SLIVER    19   too small, skipped
  TODO       0   not yet submitted
  ----------------------------------
  TOTAL    661

  progress: 97.1 %

CELL 8 COMPLETE
   State tracking defined
   Resuming is now safe

>>> RUN CELL 9 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 10: Submit function
# ==============================================================
#
# Defines only. Nothing is submitted until Cell 11.
#
# Safe to run repeatedly: only tiles in TODO are submitted.
# DONE, ACTIVE, FAILED and SLIVER are all skipped, so a restart
# never duplicates work.

def log_event(rec):
    hist = []
    if os.path.exists(LOG_FILE):
        try:
            with open(LOG_FILE) as fh:
                hist = json.load(fh)
        except Exception:
            hist = []
    hist.append(rec)
    with open(LOG_FILE, 'w') as fh:
        json.dump(hist, fh, indent=1)


def submit(limit=None, tiles=None):
    """limit  caps how many tiles this run
       tiles  overrides the selection with an explicit list"""
    st = get_state(verbose=False)
    targets = sorted(tiles) if tiles is not None else sorted(st['todo'])
    if limit:
        targets = targets[:limit]

    if not targets:
        print("Nothing to submit. Run Cell 12 to see why.")
        return

    print("Submitting", len(targets), "tiles, max",
          CONFIG['MAX_CONCURRENT'], "concurrent")
    print("")

    ok = skip = err = 0
    for k, i in enumerate(targets):
        # throttle
        while True:
            try:
                act = len({j for j, s in task_states().items()
                           if s in ('PENDING', 'RUNNING')})
            except Exception:
                act = 0
            if act < CONFIG['MAX_CONCURRENT']:
                break
            print("   ", act, "active — waiting", CONFIG['POLL_SECONDS'], "s")
            time.sleep(CONFIG['POLL_SECONDS'])

        tag = "[" + str(k + 1) + "/" + str(len(targets)) + "]"
        try:
            task, info = build_tile_task(i)
            if task is None:
                print("  ", tag, "tile", i, "skipped:", info)
                skip += 1
                log_event({'t': datetime.utcnow().isoformat(), 'tile': i,
                           'result': 'skipped', 'reason': str(info)})
                continue
            task.start()
            ok += 1
            print("  ", tag, "tile", i, "submitted,", info, "images")
            log_event({'t': datetime.utcnow().isoformat(), 'tile': i,
                       'result': 'submitted', 'n_images': info})
        except KeyboardInterrupt:
            print("")
            print("Interrupted. Re-run this cell to continue where it")
            print("stopped — nothing is lost.")
            break
        except Exception as e:
            err += 1
            print("  ", tag, "tile", i, "ERROR:", str(e)[:120])
            log_event({'t': datetime.utcnow().isoformat(), 'tile': i,
                       'result': 'error', 'message': str(e)[:300]})

    print("")
    print("submitted", ok, "  skipped", skip, "  errors", err)

print("")
print("=" * 55)
print("CELL 10 COMPLETE")
print("   submit() defined")
print("   Logging enabled")
print("")
print(">>> RUN CELL 11 NEXT")
print("=" * 55)


CELL 10 COMPLETE
   submit() defined
   Logging enabled

>>> RUN CELL 11 NEXT


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 11: Submit first 50 tiles
# ==============================================================
#
# Start small. Let these finish and inspect one before committing
# the remaining ~615 tiles.
#
# Safe to interrupt with Ctrl-C — re-running continues from where
# it stopped. Already-submitted and completed tiles are skipped.

submit(limit=50)

print("")
print("=" * 55)
print("CELL 11 COMPLETE")
print("   First batch submitted")
print("   Wait for these to finish (a few hours)")
print("   Then run Cell 12 to check progress")
print("=" * 55)

Submitting 50 tiles, max 40 concurrent



/tmp/ipykernel_4812/2383672197.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_date':         datetime.utcnow().strftime('%Y-%m-%d'),


   [1/50] tile 1 submitted, 1311 images


/tmp/ipykernel_4812/1596517025.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  log_event({'t': datetime.utcnow().isoformat(), 'tile': i,


   [2/50] tile 2 submitted, 1312 images
   [3/50] tile 3 submitted, 1322 images
   [4/50] tile 5 submitted, 1311 images
   [5/50] tile 6 submitted, 1312 images
   [6/50] tile 7 submitted, 1323 images
   [7/50] tile 10 submitted, 1310 images
   [8/50] tile 11 submitted, 1310 images
   [9/50] tile 12 submitted, 1311 images
   [10/50] tile 13 submitted, 1379 images
   [11/50] tile 14 submitted, 1463 images
   [12/50] tile 15 submitted, 1312 images
   [13/50] tile 16 submitted, 1500 images
   [14/50] tile 17 submitted, 1604 images
   [15/50] tile 18 submitted, 1662 images
   [16/50] tile 19 submitted, 1310 images
   [17/50] tile 20 submitted, 1310 images
   [18/50] tile 21 submitted, 1491 images
   [19/50] tile 22 submitted, 1547 images
   [20/50] tile 23 submitted, 1603 images
   [21/50] tile 24 submitted, 1312 images
   [22/50] tile 25 submitted, 1927 images
   [23/50] tile 26 submitted, 1998 images
   [24/50] tile 27 submitted, 2581 images
   [25/50] tile 28 submitted, 1923 images
   [2

    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
   [42/50] tile 45 submitted, 2654 images
    40 active — waiting 120 s
   [43/50] tile 46 submitted, 2698 images
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
   [44/50] tile 47 submitted, 2093 images
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
   [45/50] tile 48 submitted, 1877 images
    40 active — waiting 120 s
   [46/50] tile 49 submitted, 1879 images
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 act

   [47/50] tile 50 submitted, 1897 images
   [48/50] tile 51 submitted, 1432 images
    40 active — waiting 120 s
   [49/50] tile 52 submitted, 1424 images
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
    40 active — waiting 120 s
   [50/50] tile 53 submitted, 1425 images

submitted 50   skipped 0   errors 0

CELL 11 COMPLETE
   First batch submitted
   Wait for these to finish (a few hours)
   Then run Cell 12 to check progress


In [ ]:
# ==============================================================
#     CCDC v5.7 — HUC8 WATERSHED (667 TILES)
#     Cell 12: Check progress
# ==============================================================
#
# Run this any time, especially after a disconnect.
#
# DONE    asset exists
# ACTIVE  task running, no asset yet  <- why resuming is safe
# FAILED  task errored
# SLIVER  below MIN_TILE_FRACTION, deliberately skipped
# TODO    not yet submitted

state = get_state()

print("")
print("=" * 55)
print("CELL 12 COMPLETE")
print("   If TODO > 0, run Cell 13")
print("   If FAILED > 0, run Cell 15")
print("   If both are 0, the batch is finished")
print("=" * 55)

  DONE     642   asset exists
  ACTIVE     0   task running, no asset yet
  FAILED     0   task errored
  SLIVER    19   too small, skipped
  TODO       0   not yet submitted
  ----------------------------------
  TOTAL    661

  progress: 97.1 %

CELL 12 COMPLETE
   If TODO > 0, run Cell 13
   If FAILED > 0, run Cell 15
   If both are 0, the batch is finished


In [ ]:
# ==============================================================
#     Cell 13: Submit all remaining tiles
# ==============================================================
#
# Resumable and interruptible. Re-running picks up exactly where
# it stopped.
#
# This runs for hours. Colab may disconnect — that is fine, the
# tasks continue on Google's servers. To resume: reconnect,
# re-run Cells 1-8, then this cell.
#
# Ctrl-C stops it cleanly. Nothing is lost.

submit()

print("")
print("=" * 55)
print("CELL 13 COMPLETE")
print("   Submission pass complete")
print("   Re-run this cell if TODO remains")
print("   Then run Cell 14 to monitor")
print("=" * 55)

Nothing to submit. Run Cell 12 to see why.

CELL 13 COMPLETE
   Submission pass complete
   Re-run this cell if TODO remains
   Then run Cell 14 to monitor


In [ ]:
# ==============================================================
#     Cell 14: Monitor progress
# ==============================================================
#
# Polls every 5 minutes and prints an ETA. Interrupt any time —
# the tasks keep running regardless of whether this is open.

def monitor(interval=300, max_hours=72):
    t0 = time.time()
    start_done = len(completed_tiles())
    try:
        while time.time() - t0 < max_hours * 3600:
            st = get_state(verbose=False)
            el = (time.time() - t0) / 3600
            gained = len(st['done']) - start_done
            rate = gained / max(el, 0.01)
            left = len(st['todo']) + len(st['active'])
            eta = round(left / rate, 1) if rate > 0.01 else '?'
            print(datetime.now().strftime('[%H:%M]'),
                  "done", len(st['done']), "/", st['n'],
                  "(" + str(round(100 * len(st['done']) / st['n'], 1)) + "%)",
                  " active", len(st['active']),
                  " failed", len(st['failed']),
                  " elapsed", round(el, 1), "h",
                  " eta", eta, "h")
            if not st['todo'] and not st['active']:
                print("")
                print("COMPLETE — nothing pending")
                if st['failed']:
                    print(len(st['failed']), "failed — run Cell 15")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("")
        print("Monitoring stopped. Tasks continue running on GEE.")


monitor()

print("")
print("=" * 55)
print("CELL 14 COMPLETE")
print("   Run Cell 15 if any failed")
print("=" * 55)

[12:53] done 642 / 661 (97.1%)  active 0  failed 0  elapsed 0.0 h  eta ? h

COMPLETE — nothing pending

CELL 14 COMPLETE
   Run Cell 15 if any failed


In [ ]:
# ==============================================================
#     Cell 15: Retry failed tiles
# ==============================================================
#
# Most failures are transient memory limits or timeouts and
# succeed on a second attempt. Run this, then Cell 12 to confirm.
#
# If a tile fails repeatedly, it is likely genuinely too heavy —
# check its image count and consider processing it alone.

def retry_failed():
    st = get_state(verbose=False)
    if not st['failed']:
        print("No failed tiles.")
        return
    print("Retrying", len(st['failed']), "tiles:", sorted(st['failed']))
    print("")
    submit(tiles=sorted(st['failed']))


retry_failed()

print("")
print("=" * 55)
print("CELL 15 COMPLETE")
print("   Run Cell 12 to confirm")
print("   When TODO and FAILED are both 0, the batch is done")
print("=" * 55)

In [ ]:
# ==============================================================
#     Cell 16: Final summary — run when the batch is complete
# ==============================================================
#
# Produces numbers you will need for the methods section.

st = get_state(verbose=False)

print("=" * 55)
print("BATCH SUMMARY")
print("=" * 55)
print("")
print("Watershed area   :", round(_cache['area']), "km2")
print("Tile size        :", round(_cache['size']), "m")
print("Tiles in grid    :", st['n'])
print("Tiles processed  :", len(st['done']))
print("Tiles skipped    :", len(st['sliver']), "(sliver, below",
      str(round(100 * CONFIG['MIN_TILE_FRACTION'])) + "% of a full tile)")
print("Tiles failed     :", len(st['failed']))
print("Still pending    :", len(st['todo']) + len(st['active']))
print("")

# excluded area — a number reviewers ask for
areas = tile_areas()
full = (_cache['size'] / 1000) ** 2
excl = sum(areas[i] for i in st['sliver'])
tot = sum(areas)
print("Area coverage")
print("  processed :", round(tot - excl), "km2")
print("  excluded  :", round(excl, 1), "km2  (",
      round(100 * excl / tot, 3), "% )")
print("")

# observation density across the run
if os.path.exists(LOG_FILE):
    with open(LOG_FILE) as fh:
        hist = json.load(fh)
    counts = [h['n_images'] for h in hist
              if h.get('result') == 'submitted' and 'n_images' in h]
    if counts:
        counts.sort()
        print("Landsat images per tile")
        print("  min    :", counts[0])
        print("  median :", counts[len(counts)//2])
        print("  max    :", counts[-1])
        print("  cap    :", CONFIG['MAX_IMAGES_PER_TILE'])
        over = sum(1 for c in counts if c > CONFIG['MAX_IMAGES_PER_TILE'])
        print("  subsampled tiles:", over)
        if over == 0:
            print("")
            print("  NO TILE WAS SUBSAMPLED. Temporal density is therefore")
            print("  uniform across tile boundaries, removing the only")
            print("  mechanism by which tile membership could affect a")
            print("  result. Worth stating in the methods.")

print("")
print("=" * 55)
print("NEXT STEPS")
print("=" * 55)
print("1. Build the mosaic (createMosaic in the Code Editor)")
print("2. Tile-boundary check — does disturbance rate step at seams?")
print("3. Recompute tile 405 statistics (it was subsampled in the JS run)")
print("4. HFI accuracy assessment, Olofsson et al. (2014), with CIs")
print("5. NBAC fire validation using the corrected metrics")
print("6. False-positive rate — breaks outside both HFI and NBAC")
print("7. Threshold sensitivity for the REL_ constants")
print("8. Publication statistics at 30 m, not the 300 m diagnostics")
print("9. Archive with a Zenodo DOI")
print("=" * 55)

## Figure 3a — real Landsat/CCDC trajectories

These cells **do not alter the production CCDC assets**. They reuse the notebook's existing `load_collection()`, `run_ccdc()`, and `extract()` functions to evaluate only two 30-m locations for Figure 3a.

**Before running:** replace the two example longitude/latitude pairs in the next cell with one representative vegetation-clearing pixel and one representative oil-sands industrial pixel. Prefer interior pixels away from polygon edges. The cells print the actual CCDC change count, segment count, final-segment reliability, RMSE, and observation count so the examples can be checked against the group-level distributions before they are used in the paper.

The resulting plot uses:
- grey points = real cloud/snow-screened Landsat NDVI observations;
- coloured curves = the actual CCDC harmonic model reconstructed from `NDVI_coefs`;
- dashed vertical lines = actual CCDC break dates.


In [ ]:
# ==============================================================
# FIGURE 3a — CELL A: choose TWO real 30-m example pixels
# ==============================================================
# IMPORTANT: replace these example coordinates with your real points.
# Coordinates are longitude, latitude (EPSG:4326).
# Choose points INSIDE the HFI polygons, preferably >= 30 m from edges.

FIG3_POINTS = {
    'clearing': {
        'label': 'Vegetation clearing',
        'lon': -111.50,   # <-- REPLACE
        'lat': 56.50      # <-- REPLACE
    },
    'oilsands': {
        'label': 'Oil-sands industrial',
        'lon': -111.35,   # <-- REPLACE
        'lat': 57.00      # <-- REPLACE
    }
}

print('Figure 3a points:')
for k, v in FIG3_POINTS.items():
    print(f"  {k:10s}: ({v['lon']:.6f}, {v['lat']:.6f})  {v['label']}")
print('\nReplace the coordinates above before running Cell B.')


In [ ]:
# ==============================================================
# FIGURE 3a — CELL B: extract real observations + raw CCDC arrays
# ==============================================================
import math
import pandas as pd
import numpy as np
from datetime import datetime, timezone

FIG3_SCALE = 30
FIG3_CRS = CONFIG['EXPORT_CRS']


def _point_value(img, band, point):
    """First valid 30-m pixel value at a point."""
    return img.select(band).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=FIG3_SCALE,
        crs=FIG3_CRS,
        maxPixels=10000
    ).get(band)


def _observations_at_point(col, point):
    """One feature per real Landsat NDVI observation at the point."""
    def to_feature(img):
        ms = ee.Number(img.get('system:time_start'))
        val = _point_value(img, 'NDVI', point)
        return ee.Feature(None, {
            'millis': ms,
            'date': ee.Date(ms).format('YYYY-MM-dd'),
            'NDVI': val
        })

    fc = ee.FeatureCollection(col.map(to_feature)).filter(
        ee.Filter.notNull(['NDVI'])
    )
    rows = fc.getInfo()['features']
    out = pd.DataFrame([f['properties'] for f in rows])
    if len(out):
        out['millis'] = pd.to_numeric(out['millis'])
        out['NDVI'] = pd.to_numeric(out['NDVI'])
        out['date'] = pd.to_datetime(out['date'])
        out = out.sort_values('millis').reset_index(drop=True)
    return out


def _raw_ccdc_at_point(ccdc, point):
    """Retrieve the raw CCDC arrays needed to reconstruct NDVI fits."""
    bands = [
        'tStart', 'tEnd', 'tBreak', 'numObs', 'changeProb',
        'NDVI_coefs', 'NDVI_rmse', 'NDVI_magnitude'
    ]
    raw = ccdc.select(bands).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=FIG3_SCALE,
        crs=FIG3_CRS,
        maxPixels=10000
    ).getInfo()
    return raw


def _summary_at_point(ccdc, point):
    """Use the exact 39-band extraction logic from the production notebook."""
    small_geom = point.buffer(45)
    s = extract(ccdc, small_geom).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=FIG3_SCALE,
        crs=FIG3_CRS,
        maxPixels=10000
    ).getInfo()
    keep = [
        'change_count', 'segment_count', 'change_frequency_per_decade',
        'first_break_year', 'last_break_year', 'last_segment_start',
        'last_segment_end', 'last_segment_duration_years',
        'ndvi_slope_last', 'ndvi_rmse_last', 'num_obs_last',
        'num_obs_total', 'last_segment_reliable', 'qa_flags'
    ]
    return {k: s.get(k) for k in keep}


def extract_real_ccdc_example(name, info):
    point = ee.Geometry.Point([info['lon'], info['lat']])
    # Small search geometry is enough; CCDC is evaluated lazily at the point.
    roi = point.buffer(90)

    col = load_collection(roi).select(INPUT_BANDS)
    n_images = col.size().getInfo()
    if n_images < CONFIG['MIN_OBSERVATIONS']:
        raise RuntimeError(f'{name}: only {n_images} input images.')

    ccdc = run_ccdc(col)
    obs = _observations_at_point(col, point)
    raw = _raw_ccdc_at_point(ccdc, point)
    summary = _summary_at_point(ccdc, point)

    return {
        'name': name,
        'label': info['label'],
        'lon': info['lon'],
        'lat': info['lat'],
        'n_collection_images': n_images,
        'observations': obs,
        'raw': raw,
        'summary': summary
    }


FIG3_REAL = {}
for name, info in FIG3_POINTS.items():
    print('\n' + '=' * 62)
    print('EXTRACTING:', info['label'])
    print('=' * 62)
    ex = extract_real_ccdc_example(name, info)
    FIG3_REAL[name] = ex
    print('Collection images :', ex['n_collection_images'])
    print('Valid NDVI obs    :', len(ex['observations']))
    for k, v in ex['summary'].items():
        print(f'{k:30s}: {v}')

print('\nExtraction complete.')


In [ ]:
# ==============================================================
# FIGURE 3a — CELL C: reconstruct ACTUAL CCDC NDVI fitted segments
# ==============================================================
# CCDC coefficient order:
# [intercept, slope, cos(1ωt), sin(1ωt), cos(2ωt), sin(2ωt),
#  cos(3ωt), sin(3ωt)]
# With dateFormat=2, t is Unix milliseconds, so the annual angular
# frequency is 2*pi / milliseconds_per_year.

MILLISECONDS_PER_YEAR = 31557600000.0
OMEGA_MS = 2.0 * math.pi / MILLISECONDS_PER_YEAR


def ccdc_ndvi_value(t_ms, coef):
    """Evaluate one CCDC NDVI harmonic model row at Unix time t_ms."""
    c = np.asarray(coef, dtype='float64')
    if c.size < 8:
        raise ValueError(f'Expected 8 NDVI coefficients, got {c.size}')
    return (
        c[0] + c[1] * t_ms
        + c[2] * np.cos(OMEGA_MS * t_ms)
        + c[3] * np.sin(OMEGA_MS * t_ms)
        + c[4] * np.cos(2.0 * OMEGA_MS * t_ms)
        + c[5] * np.sin(2.0 * OMEGA_MS * t_ms)
        + c[6] * np.cos(3.0 * OMEGA_MS * t_ms)
        + c[7] * np.sin(3.0 * OMEGA_MS * t_ms)
    )


def millis_to_datetime(ms):
    return pd.to_datetime(float(ms), unit='ms', utc=True).tz_convert(None)


def build_fitted_segments(example, points_per_year=24):
    raw = example['raw']
    starts = raw.get('tStart') or []
    ends   = raw.get('tEnd') or []
    breaks = raw.get('tBreak') or []
    coefs  = raw.get('NDVI_coefs') or []

    nseg = min(len(starts), len(ends), len(coefs))
    fits = []
    for i in range(nseg):
        t0 = float(starts[i])
        t1 = float(ends[i])
        if not np.isfinite(t0) or not np.isfinite(t1) or t1 <= t0:
            continue
        years = max((t1 - t0) / MILLISECONDS_PER_YEAR, 0.1)
        n = max(8, int(np.ceil(years * points_per_year)))
        tt = np.linspace(t0, t1, n)
        yy = np.array([ccdc_ndvi_value(x, coefs[i]) for x in tt])
        fits.append(pd.DataFrame({
            'segment': i + 1,
            'millis': tt,
            'date': [millis_to_datetime(x) for x in tt],
            'fitted_NDVI': yy
        }))

    fit = pd.concat(fits, ignore_index=True) if fits else pd.DataFrame()
    real_breaks = [float(x) for x in breaks if x is not None and float(x) > 0]
    break_dates = [millis_to_datetime(x) for x in real_breaks]
    return fit, break_dates


for name, ex in FIG3_REAL.items():
    fit, bdates = build_fitted_segments(ex)
    ex['fit'] = fit
    ex['break_dates'] = bdates
    print('\n', ex['label'])
    print('  fitted segments:', fit['segment'].nunique() if len(fit) else 0)
    print('  real breaks    :', [d.strftime('%Y-%m-%d') for d in bdates])
    if len(fit):
        print('  fitted NDVI range:', round(fit.fitted_NDVI.min(), 3),
              'to', round(fit.fitted_NDVI.max(), 3))


In [ ]:
# ==============================================================
# FIGURE 3a — CELL D: publication plot + local CSV exports
# ==============================================================
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path('/content/Fig3a_real')
OUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    'clearing': '#2E7D32',
    'oilsands': '#B5461F'
}

# ---- Save the numerical evidence used by the figure ----
for name, ex in FIG3_REAL.items():
    obs = ex['observations'].copy()
    obs['group'] = name
    obs['lon'] = ex['lon']
    obs['lat'] = ex['lat']
    obs.to_csv(OUT_DIR / f'{name}_observed_NDVI.csv', index=False)

    fit = ex['fit'].copy()
    fit['group'] = name
    fit['lon'] = ex['lon']
    fit['lat'] = ex['lat']
    fit.to_csv(OUT_DIR / f'{name}_CCDC_fitted_NDVI.csv', index=False)

    pd.DataFrame({
        'break_date': ex['break_dates']
    }).to_csv(OUT_DIR / f'{name}_CCDC_breaks.csv', index=False)

    pd.DataFrame([ex['summary']]).to_csv(
        OUT_DIR / f'{name}_CCDC_summary.csv', index=False)

# ---- Plot ----
fig, axes = plt.subplots(2, 1, figsize=(9.0, 6.3), sharex=True)

order = ['clearing', 'oilsands']
titles = {
    'clearing': '(a) Vegetation clearing — real example',
    'oilsands': '(b) Oil-sands industrial — real example'
}

for ax, name in zip(axes, order):
    ex = FIG3_REAL[name]
    obs = ex['observations']
    fit = ex['fit']
    color = COLORS[name]

    # Actual observations
    ax.scatter(obs['date'], obs['NDVI'], s=7, alpha=0.28,
               color='#87949B', edgecolors='none',
               label='Observed Landsat NDVI')

    # Actual fitted CCDC segments
    for seg, ss in fit.groupby('segment'):
        ax.plot(ss['date'], ss['fitted_NDVI'], color=color,
                linewidth=2.0, zorder=3)

    # Actual break dates
    for bd in ex['break_dates']:
        ax.axvline(bd, color='0.35', linestyle='--', linewidth=0.8,
                   alpha=0.75)

    sm = ex['summary']
    reliability = int(sm['last_segment_reliable']) if sm['last_segment_reliable'] is not None else -1
    txt = (
        f"CCDC breaks = {int(sm['change_count']) if sm['change_count'] is not None else 'NA'}   |   "
        f"segments = {int(sm['segment_count']) if sm['segment_count'] is not None else 'NA'}   |   "
        f"final reliable = {reliability}"
    )

    ax.text(0.015, 0.03, txt, transform=ax.transAxes, fontsize=9,
            ha='left', va='bottom',
            bbox=dict(facecolor='white', edgecolor='0.75', alpha=0.88,
                      boxstyle='round,pad=0.25'))

    ax.set_title(titles[name], loc='left', fontsize=12, fontweight='bold')
    ax.set_ylabel('NDVI')
    ax.set_ylim(-0.15, 1.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].set_xlabel('Year')
fig.suptitle('Representative real Landsat observations and CCDC fitted trajectories',
             fontsize=13, fontweight='bold', x=0.08, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.965])

png = OUT_DIR / 'Fig3a_REAL_CCDC_trajectories.png'
pdf = OUT_DIR / 'Fig3a_REAL_CCDC_trajectories.pdf'
fig.savefig(png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(pdf, bbox_inches='tight', facecolor='white')
plt.show()

print('\nSaved to:', OUT_DIR)
print(' ', png)
print(' ', pdf)
print('\nCSV files were also saved so every plotted value is auditable.')


In [ ]:
# ==============================================================
# FIGURE 3a — CELL E: optional ZIP for download from Colab
# ==============================================================
import shutil
zip_path = shutil.make_archive('/content/Fig3a_real_outputs', 'zip', '/content/Fig3a_real')
print('ZIP created:', zip_path)

# In Colab, uncomment these two lines to download the ZIP directly:
# from google.colab import files
# files.download(zip_path)
